# Accident video classifier — ResNet18 + LSTM on short dense windows

**Why this design**
- A collision lasts only 1–2 s of a ~40 s video. The old model squeezed 32 frames out of the whole video (about 1 frame every 1.25 s), so the event was almost never visible.
- This model looks at **short dense windows** (16 consecutive frames = 5.3 s at 3 fps). An ImageNet-pretrained **ResNet18** encodes each frame and an **LSTM** models the motion across the window.
- **Training:** positive windows are taken around the event (`time_of_event` is used *only* to choose where to crop in training videos; it is never a model input). Negative windows are random and also aligned to typical event times, so window position is not a shortcut.
- **Validation/inference:** slide the window over the whole video and use the mean of the top-3 window logits as the video score. No metadata is used.

**Fixes in this version (v2)**
- No checkpoint selection on the 60 validation clips (that inflated the val AUC). Fixed 8 epochs, weights = EMA of the training run, val is monitor-only.
- Less overfitting: only `layer4` trained, backbone LR 5e-5, dropout on per-frame features, LSTM hidden 128, label smoothing 0.1, LR schedule now actually finishes (8-epoch cosine).
- Training/eval mismatch: positive windows now place the event anywhere inside the window (the sliding-window evaluation sees it at every position); positives without an event time use a typical event time instead of a random (possibly event-free) window.
- Ensemble of `N_SEEDS` models (averaged logits) for a more stable score.

Input: `data/processed_v3/{train_240,val_60}.csv` and `data/sampled_frames_3fps_224/*.npz` from `preprocess.ipynb`.

In [1]:
import copy, math, random
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve,
                             classification_report, confusion_matrix)
from tqdm.auto import tqdm

# ---------------- configuration ----------------
SEED          = 42
PROCESSED_DIR = Path("data/processed_v3")
TRAIN_CSV     = PROCESSED_DIR / "train_240.csv"
VAL_CSV       = PROCESSED_DIR / "val_60.csv"
FRAMES_DIR    = Path("data/sampled_frames_3fps_224")

WIN           = 16      # frames per window (16 / 3 fps = 5.3 s)
EVAL_STRIDE   = 4       # sliding-window step at validation (frames)
TOPK          = 3       # video score = mean of top-k window logits
CROP          = 160     # spatial size fed to the network (use 112 if CUDA out of memory)

BATCH_SIZE        = 8
SAMPLES_PER_VIDEO = 2   # windows drawn per training video per epoch
EPOCHS            = 3   # fixed cap chosen from training-curve shape (val overfit after ~2-3 ep); not selected on val
FREEZE_BACKBONE   = False  # True: train only LSTM + head (safest with 240 clips)
WARMUP_EPOCHS     = 1
SEEDS             = [42, 43, 44]   # ensemble size (use [42] for a quick run)
EMA_DECAY         = 0.98   # ~1 epoch horizon
FEAT_DROP         = 0.5    # dropout on per-frame features
LR_BACKBONE       = 5e-5
LR_HEAD           = 1e-3
WD                = 1e-2
LABEL_SMOOTH      = 0.1
DROPOUT           = 0.5
TRAINABLE_BLOCKS  = ("layer4",)   # everything before layer4 stays frozen

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
CKPT    = "resnet18_lstm_seed{}.pth"
LSTM_HIDDEN = 128

# ImageNet normalisation (ResNet18 weights)
K_MEAN = (0.485, 0.456, 0.406)
K_STD  = (0.229, 0.224, 0.225)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE.type != "cuda":
    print("WARNING: no GPU - training will be very slow.")

device: cuda | torch 2.11.0+cu128


c:\Users\Mobin-pc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------- splits & sanity checks ----------------
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

assert set(train_df["video_path"]).isdisjoint(val_df["video_path"]), "Data leakage: overlapping videos"

def npz_path(video_path):
    return FRAMES_DIR / Path(video_path).with_suffix(".npz")

missing = [p for p in pd.concat([train_df, val_df])["video_path"] if not npz_path(p).exists()]
assert not missing, f"{len(missing)} npz files missing, e.g. {missing[:3]}"

def load_frames(video_path):
    with np.load(npz_path(video_path), allow_pickle=False) as z:
        return z["frames"], z["timestamps"].astype(np.float64)   # (T,224,224,3) uint8, seconds

# time_of_event: used ONLY to choose training crops for positive videos.
train_event = train_df["time_of_event"].to_numpy(dtype=float)
POS_EVENT_TIMES = train_event[(train_df["label"] == 1).to_numpy() & np.isfinite(train_event)]

print(f"train {len(train_df)} {train_df['label'].value_counts().to_dict()} | "
      f"val {len(val_df)} {val_df['label'].value_counts().to_dict()}")
print(f"positive train videos with event time: {len(POS_EVENT_TIMES)} / {(train_df['label'] == 1).sum()}")
if len(POS_EVENT_TIMES):
    print(f"event time (s): min {POS_EVENT_TIMES.min():.1f} | median {np.median(POS_EVENT_TIMES):.1f} "
          f"| max {POS_EVENT_TIMES.max():.1f}")

train 240 {0: 120, 1: 120} | val 60 {1: 30, 0: 30}
positive train videos with event time: 120 / 120
event time (s): min 9.3 | median 19.8 | max 56.8


In [3]:
# ---------------- window sampling, loader, GPU augmentation ----------------
def window_indices(n, start):
    # WIN consecutive frames; very short videos repeat their last frame
    return np.clip(np.arange(start, start + WIN), 0, n - 1)


def start_around(ts, t_event, n):
    # put the event in the second half of the window (some context before, a little after)
    e = int(np.argmin(np.abs(ts - t_event)))
    s = e - np.random.randint(2, WIN - 2)   # event anywhere inside the window, like sliding-window eval
    return int(np.clip(s, 0, max(n - WIN, 0)))


class TrainWindows:
    def __init__(self, df):
        self.paths  = df["video_path"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.events = df["time_of_event"].to_numpy(dtype=float)

    def __len__(self):
        return len(self.paths) * SAMPLES_PER_VIDEO

    def __getitem__(self, i):
        j = i % len(self.paths)
        frames, ts = load_frames(self.paths[j])
        n, label, t_ev = len(frames), self.labels[j], self.events[j]
        if label == 1:
            # no event time recorded -> typical event time, never a random (event-free) window
            s = start_around(ts, t_ev if np.isfinite(t_ev) else np.random.choice(POS_EVENT_TIMES), n)
        elif label == 0 and len(POS_EVENT_TIMES) and np.random.rand() < 0.5:
            # same temporal position as a real event -> position is not a shortcut
            s = start_around(ts, np.random.choice(POS_EVENT_TIMES), n)
        else:
            s = np.random.randint(0, max(n - WIN, 0) + 1)
        clip = np.ascontiguousarray(frames[window_indices(n, s)])
        return torch.from_numpy(clip), torch.tensor(float(label))


class ThreadLoader:
    # Thread-based loader (no multiprocessing -> no Windows worker crashes)
    def __init__(self, dataset, batch_size, shuffle=True, drop_last=True, threads=4, prefetch=3):
        self.ds, self.bs, self.shuffle, self.drop_last = dataset, batch_size, shuffle, drop_last
        self.prefetch = prefetch
        self.pool = ThreadPoolExecutor(max_workers=threads)

    def __len__(self):
        n = len(self.ds)
        return n // self.bs if self.drop_last else -(-n // self.bs)

    def _make(self, ids):
        items = list(self.pool.map(self.ds.__getitem__, ids))
        clips = torch.stack([c for c, _ in items])
        labels = torch.stack([l for _, l in items])
        if DEVICE.type == "cuda":
            clips, labels = clips.pin_memory(), labels.pin_memory()
        return clips, labels

    def __iter__(self):
        order = np.random.permutation(len(self.ds)) if self.shuffle else np.arange(len(self.ds))
        batches = iter([order[k * self.bs:(k + 1) * self.bs] for k in range(len(self))])
        q = deque()
        with ThreadPoolExecutor(max_workers=1) as assembler:
            for ids in batches:
                q.append(assembler.submit(self._make, ids))
                if len(q) >= self.prefetch:
                    break
            while q:
                batch = q.popleft().result()
                nxt = next(batches, None)
                if nxt is not None:
                    q.append(assembler.submit(self._make, nxt))
                yield batch


_MEAN = torch.tensor(K_MEAN, device=DEVICE).view(1, 1, 3, 1, 1)
_STD  = torch.tensor(K_STD,  device=DEVICE).view(1, 1, 3, 1, 1)


def gpu_prepare(clips, train):
    # uint8 (B,T,H,W,C) on GPU -> normalised float (B,T,3,CROP,CROP)
    x = clips.permute(0, 1, 4, 2, 3).float().div_(255)        # B,T,C,H,W
    B, T, C, H, W = x.shape
    if train:
        out = []
        for c in x:                                            # same augmentation for all frames of a clip
            side = int(min(H, W) * random.uniform(0.8, 1.0))
            i = random.randint(0, H - side); j = random.randint(0, W - side)
            c = TF.resized_crop(c, i, j, side, side, [CROP, CROP], antialias=True)
            if random.random() < 0.5:
                c = c.flip(-1)
            c = TF.adjust_brightness(c, random.uniform(0.8, 1.2))
            c = TF.adjust_contrast(c, random.uniform(0.8, 1.2))
            c = TF.adjust_saturation(c, random.uniform(0.8, 1.2))
            out.append(c.clamp_(0, 1))
        x = torch.stack(out)
    else:
        x = F.interpolate(x.reshape(B * T, C, H, W), size=(CROP, CROP),
                          mode="bilinear", align_corners=False, antialias=True).reshape(B, T, C, CROP, CROP)
    return (x - _MEAN) / _STD


train_loader = ThreadLoader(TrainWindows(train_df), BATCH_SIZE, shuffle=True, drop_last=True)
print(f"train batches / epoch: {len(train_loader)}")

train batches / epoch: 60


In [4]:
# ---------------- model ----------------
class ResNetLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        bb = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        bb.fc = nn.Identity()                                  # 512-d per-frame feature
        self.backbone = bb
        self.lstm = nn.LSTM(512, LSTM_HIDDEN, num_layers=1, batch_first=True)
        self.feat_drop = nn.Dropout(FEAT_DROP)
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(2 * LSTM_HIDDEN, 1))
        for name, p in self.backbone.named_parameters():
            p.requires_grad = (not FREEZE_BACKBONE) and name.split(".")[0] in TRAINABLE_BLOCKS

    def train(self, mode=True):
        super().train(mode)
        # small batches: keep pretrained BatchNorm statistics fixed
        for m in self.backbone.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()
        return self

    def forward(self, x):                                      # x: B,T,3,H,W
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).view(B, T, -1)      # B,T,512
        f = self.feat_drop(f)
        out, _ = self.lstm(f)                                  # B,T,H
        z = torch.cat([out[:, -1], out.mean(1)], dim=1)        # last state + temporal mean
        return self.head(z).squeeze(1)


model = ResNetLSTM().to(DEVICE)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all   = sum(p.numel() for p in model.parameters())
print(f"trainable params: {n_train / 1e6:.1f}M / {n_all / 1e6:.1f}M")

trainable params: 8.7M / 11.5M


In [5]:
# ---------------- video-level evaluation (sliding windows, no metadata) ----------------
def window_starts(n):
    last = max(n - WIN, 0)
    starts = list(range(0, last + 1, EVAL_STRIDE))
    if starts[-1] != last:
        starts.append(last)
    return starts


def as_list(m):
    return list(m) if isinstance(m, (list, tuple)) else [m]


@torch.no_grad()
def window_logits(models, frames, tta=False, chunk=16):
    models = as_list(models)
    starts = window_starts(len(frames))
    idx = np.stack([window_indices(len(frames), s) for s in starts])
    clips = torch.from_numpy(frames[idx])                      # Nw,WIN,H,W,C uint8
    out = []
    for k in range(0, len(starts), chunk):
        x = gpu_prepare(clips[k:k + chunk].to(DEVICE, non_blocking=True), train=False)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            l = 0
            for m in models:
                lm = m(x).float()
                if tta:
                    lm = 0.5 * (lm + m(x.flip(-1)).float())
                l = l + lm / len(models)
        out.append(l.cpu())
    return torch.cat(out).numpy(), np.array(starts)


def video_score(logits):
    k = min(TOPK, len(logits))
    return float(np.sort(logits)[-k:].mean())


def evaluate(model, df, tta=False, prefetch=3):
    for m in as_list(model):
        m.eval()
    paths = df["video_path"].tolist()
    y = df["label"].astype(int).to_numpy()
    scores, peak_time = [], []
    with ThreadPoolExecutor(max_workers=prefetch) as pool:
        futs = [pool.submit(load_frames, p) for p in paths[:prefetch]]
        for k in range(len(paths)):
            frames, ts = futs[k].result(); futs[k] = None
            if k + prefetch < len(paths):
                futs.append(pool.submit(load_frames, paths[k + prefetch]))
            logits, starts = window_logits(model, frames, tta=tta)
            scores.append(video_score(logits))
            s = starts[int(np.argmax(logits))]
            peak_time.append(float(ts[min(s + WIN - 1, len(ts) - 1)]))   # end of most suspicious window
    scores = np.array(scores)
    prob = 1 / (1 + np.exp(-scores))
    pred = (prob >= 0.5).astype(int)
    loss = float(F.binary_cross_entropy_with_logits(torch.tensor(scores), torch.tensor(y, dtype=torch.float64)))
    return {
        "loss": loss,
        "auc":  roc_auc_score(y, prob) if len(np.unique(y)) > 1 else float("nan"),
        "acc":  accuracy_score(y, pred),
        "f1":   f1_score(y, pred, zero_division=0),
        "y": y, "prob": prob, "peak_time": np.array(peak_time),
    }

In [6]:
# ---------------- training (fixed epochs, EMA weights, no checkpoint selection) ----------------
def make_ema(m):
    e = copy.deepcopy(m).eval()
    for p in e.parameters():
        p.requires_grad_(False)
    return e

@torch.no_grad()
def ema_update(ema, m, decay):
    for pe, p in zip(ema.parameters(), m.parameters()):
        pe.mul_(decay).add_(p.detach(), alpha=1 - decay)


def train_one(seed):
    seed_everything(seed)
    model = ResNetLSTM().to(DEVICE)
    ema = make_ema(model)

    optimizer = torch.optim.AdamW([
        *([{"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_BACKBONE}] if not FREEZE_BACKBONE else []),
        {"params": list(model.lstm.parameters()) + list(model.head.parameters()), "lr": LR_HEAD},
    ], weight_decay=WD)

    steps_per_epoch = len(train_loader)
    total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    criterion = nn.BCEWithLogitsLoss()
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        run_loss, n_seen, correct = 0.0, 0, 0
        for clips, labels in tqdm(train_loader, desc=f"seed {seed} epoch {epoch:02d}", leave=False):
            x = gpu_prepare(clips.to(DEVICE, non_blocking=True), train=True)
            labels = labels.to(DEVICE, non_blocking=True)
            target = labels * (1 - LABEL_SMOOTH) + 0.5 * LABEL_SMOOTH
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
            loss = criterion(logits.float(), target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema_update(ema, model, EMA_DECAY)
            run_loss += loss.item() * len(labels); n_seen += len(labels)
            correct  += ((logits.detach().float() > 0).float() == labels).sum().item()

        vm = evaluate(ema, val_df)   # MONITOR ONLY: nothing is selected or stopped on this
        history.append({"seed": seed, "epoch": epoch, "train_loss": run_loss / n_seen,
                        "train_acc": correct / n_seen, **{k: vm[k] for k in ("loss", "auc", "acc", "f1")}})
        print(f"[seed {seed}] ep {epoch:02d} | train L {run_loss / n_seen:.4f} acc {correct / n_seen:.3f} || "
              f"val(EMA) L {vm['loss']:.4f} acc {vm['acc']:.3f} AUC {vm['auc']:.4f}")

    torch.save(ema.state_dict(), CKPT.format(seed))
    return ema, history


models, all_history = [], []
for seed in SEEDS:
    ema, h = train_one(seed)
    models.append(ema); all_history += h

pd.DataFrame(all_history)


[seed 42] ep 01 | train L 0.6936 acc 0.548 || val(EMA) L 0.6667 acc 0.650 AUC 0.6811


[seed 42] ep 02 | train L 0.6392 acc 0.640 || val(EMA) L 0.6596 acc 0.600 AUC 0.6511


[seed 42] ep 03 | train L 0.5363 acc 0.771 || val(EMA) L 0.6564 acc 0.617 AUC 0.6889


[seed 43] ep 01 | train L 0.6912 acc 0.517 || val(EMA) L 0.6862 acc 0.550 AUC 0.5800


[seed 43] ep 02 | train L 0.6137 acc 0.685 || val(EMA) L 0.6516 acc 0.633 AUC 0.6767


[seed 43] ep 03 | train L 0.5458 acc 0.762 || val(EMA) L 0.6700 acc 0.633 AUC 0.7000


[seed 44] ep 01 | train L 0.6693 acc 0.590 || val(EMA) L 0.6484 acc 0.633 AUC 0.7122


[seed 44] ep 02 | train L 0.5998 acc 0.706 || val(EMA) L 0.6404 acc 0.617 AUC 0.6878


[seed 44] ep 03 | train L 0.5446 acc 0.748 || val(EMA) L 0.6637 acc 0.650 AUC 0.7022


,seed,epoch,train_loss,train_acc,loss,auc,acc,f1
0,42,1,0.693573,0.547917,0.666735,0.681111,0.650000,0.720000
1,42,2,0.639200,0.639583,0.659626,0.651111,0.600000,0.675676
2,42,3,0.536321,0.770833,0.656437,0.688889,0.616667,0.656716
3,43,1,0.691174,0.516667,0.686208,0.580000,0.550000,0.640000
4,43,2,0.613726,0.685417,0.651553,0.676667,0.633333,0.666667
5,43,3,0.545836,0.762500,0.669953,0.700000,0.633333,0.676471
6,44,1,0.669267,0.589583,0.648446,0.712222,0.633333,0.685714
7,44,2,0.599773,0.706250,0.640422,0.687778,0.616667,0.666667
8,44,3,0.544583,0.747917,0.663695,0.702222,0.650000,0.704225


In [7]:
# ---------------- final report (best checkpoint, flip TTA) ----------------
res = evaluate(models, val_df, tta=True)   # ensemble, final-epoch EMA weights
y, prob = res["y"], res["prob"]

def report(name, thr):
    pred = (prob >= thr).astype(int)
    print(f"\n===== {name}: threshold {thr:.3f} =====")
    print(classification_report(y, pred, target_names=["Negative (0)", "Positive (1)"], digits=3, zero_division=0))
    print("confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y, pred))
    return accuracy_score(y, pred)

print(f"ensemble of {len(models)} | val AUC (TTA) {res['auc']:.4f}")
acc_05  = report("fixed 0.5", 0.5)
# classes are balanced (50/50) by construction -> median score is a label-free threshold
acc_med = report("median of scores (label-free, honest)", float(np.median(prob)))
fpr, tpr, thr = roc_curve(y, prob)
acc_you = report("Youden on val (OPTIMISTIC: tuned on these labels)", float(thr[np.argmax(tpr - fpr)]))

print(f"\nSUMMARY  AUC {res['auc']:.4f} | acc@0.5 {acc_05:.3f} | acc@median {acc_med:.3f} | acc@Youden {acc_you:.3f}")
print("With 60 validation videos one video = 1.7 points; standard error of accuracy is about ±6 points.")

# Diagnostic only: does the model look at the right moment in positive videos?
pos = (y == 1) & np.isfinite(val_df["time_of_event"].to_numpy(dtype=float))
if pos.any():
    err = np.abs(res["peak_time"][pos] - val_df["time_of_event"].to_numpy(dtype=float)[pos])
    print(f"positives: most suspicious window ends within 3 s of the true event in "
          f"{(err <= 3).mean():.0%} of videos (median error {np.median(err):.1f} s)")

ensemble of 3 | val AUC (TTA) 0.6900

===== fixed 0.5: threshold 0.500 =====
              precision    recall  f1-score   support

Negative (0)      0.778     0.467     0.583        30
Positive (1)      0.619     0.867     0.722        30

    accuracy                          0.667        60
   macro avg      0.698     0.667     0.653        60
weighted avg      0.698     0.667     0.653        60

confusion matrix [[TN FP] [FN TP]]:
 [[14 16]
 [ 4 26]]

===== median of scores (label-free, honest): threshold 0.594 =====
              precision    recall  f1-score   support

Negative (0)      0.667     0.667     0.667        30
Positive (1)      0.667     0.667     0.667        30

    accuracy                          0.667        60
   macro avg      0.667     0.667     0.667        60
weighted avg      0.667     0.667     0.667        60

confusion matrix [[TN FP] [FN TP]]:
 [[20 10]
 [10 20]]

===== Youden on val (OPTIMISTIC: tuned on these labels): threshold 0.671 =====
         

In [8]:
# ---------------- inference on a new video (.npz from preprocess.ipynb) ----------------
def predict_npz(path, threshold=0.5, tta=True):
    with np.load(path, allow_pickle=False) as z:
        frames, ts = z["frames"], z["timestamps"]
    for m in models:
        m.eval()
    logits, starts = window_logits(models, frames, tta=tta)
    p = 1 / (1 + math.exp(-video_score(logits)))
    t_peak = float(ts[min(starts[int(np.argmax(logits))] + WIN - 1, len(ts) - 1)])
    return {"prob_accident": p, "label": int(p >= threshold), "most_suspicious_time_s": t_peak}

# example:
# predict_npz(npz_path(val_df["video_path"].iloc[0]))